<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Munsell_to_RGB_Lab_D65_adapted_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Munsell codes to CIE Lab (D65) and sRGB

**Notebook version 3.0 — corrected and validated JSON**

Standard Munsell renotation data are defined under CIE Illuminant C with the CIE 1931 2 degree standard observer. This notebook converts Munsell notation to xyY(C), then XYZ(C), chromatically adapts XYZ(C) to XYZ(D65) with CAT02, and finally calculates CIE Lab(D65) and encoded sRGB.


In [1]:
# Step 1: Install missing dependencies and import the required packages.
import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_PACKAGES = {
    'colour-science': 'colour',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'Pillow': 'PIL',
    'python-docx': 'docx',
}

missing = [package for package, module in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])

import colour
import numpy as np
import pandas as pd
from PIL import Image
from docx import Document
from docx.shared import Inches

try:
    from google.colab import files
except ImportError:
    files = None

OUTPUT_DIRECTORY = Path('output')
OUTPUT_DIRECTORY.mkdir(exist_ok=True)
print(f'colour-science version: {colour.__version__}')


colour-science version: 0.4.7


In [2]:
# Step 2: Define source and target illuminants for the colour conversion.
OBSERVER = 'CIE 1931 2 Degree Standard Observer'
ILLUMINANT_C_XY = colour.CCS_ILLUMINANTS[OBSERVER]['C']
ILLUMINANT_D65_XY = colour.CCS_ILLUMINANTS[OBSERVER]['D65']

# The adaptation function requires both reference whites as XYZ values.
WHITE_C_XYZ = colour.xy_to_XYZ(ILLUMINANT_C_XY)
WHITE_D65_XYZ = colour.xy_to_XYZ(ILLUMINANT_D65_XY)
CHROMATIC_ADAPTATION_TRANSFORM = 'CAT02'

print('Illuminant C xy:', ILLUMINANT_C_XY)
print('Illuminant D65 xy:', ILLUMINANT_D65_XY)
print('Adaptation transform:', CHROMATIC_ADAPTATION_TRANSFORM)


Illuminant C xy: [ 0.31006  0.31616]
Illuminant D65 xy: [ 0.3127  0.329 ]
Adaptation transform: CAT02


In [3]:
# Step 3: Convert Munsell input defined under C to Lab(D65) and sRGB.
def munsell_to_d65_colourimetry(munsell_code: str):
    """Return xyY(C), XYZ(C), XYZ(D65), Lab(D65), sRGB, and RGB8."""
    code = str(munsell_code).strip()

    # Munsell renotation lookup returns xyY under Illuminant C.
    xyY_C = colour.notation.munsell_colour_to_xyY(code)
    XYZ_C = colour.xyY_to_XYZ(xyY_C)

    # Adapt the source C-relative tristimulus values to the D65 reference white.
    XYZ_D65 = colour.adaptation.chromatic_adaptation_VonKries(
        XYZ_C,
        WHITE_C_XYZ,
        WHITE_D65_XYZ,
        transform=CHROMATIC_ADAPTATION_TRANSFORM,
    )

    # Calculate Lab with its D65 reference white explicitly specified.
    Lab_D65 = colour.XYZ_to_Lab(XYZ_D65, illuminant=ILLUMINANT_D65_XY)

    # sRGB is D65-based. Clip only the display/export RGB representation.
    RGB_unclipped = colour.XYZ_to_sRGB(XYZ_D65, apply_cctf_encoding=True)
    RGB_sRGB = np.clip(RGB_unclipped, 0.0, 1.0)
    RGB8 = np.rint(RGB_sRGB * 255.0).astype(np.uint8)

    return xyY_C, XYZ_C, XYZ_D65, Lab_D65, RGB_sRGB, RGB8


def convert_munsell_dataframe(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Convert a dataframe containing Item and Munsell code columns."""
    required = {'Item', 'Munsell code'}
    if not required.issubset(dataframe.columns):
        raise ValueError('Input must have Item and Munsell code columns.')

    rows = []
    for _, row in dataframe.iterrows():
        code = str(row['Munsell code']).strip()
        xyY_C, XYZ_C, XYZ_D65, Lab_D65, RGB_sRGB, RGB8 = munsell_to_d65_colourimetry(code)
        rows.append({
            'Item': row['Item'],
            'Munsell code': code,
            'x_C': xyY_C[0],
            'y_C': xyY_C[1],
            'Y_C': xyY_C[2],
            'X_C': XYZ_C[0],
            'Y_C_XYZ': XYZ_C[1],
            'Z_C': XYZ_C[2],
            'X_D65': XYZ_D65[0],
            'Y_D65': XYZ_D65[1],
            'Z_D65': XYZ_D65[2],
            'L*_D65': Lab_D65[0],
            'a*_D65': Lab_D65[1],
            'b*_D65': Lab_D65[2],
            'R_sRGB': int(RGB8[0]),
            'G_sRGB': int(RGB8[1]),
            'B_sRGB': int(RGB8[2]),
            'R_sRGB_normalised': RGB_sRGB[0],
            'G_sRGB_normalised': RGB_sRGB[1],
            'B_sRGB_normalised': RGB_sRGB[2],
        })

    return pd.DataFrame(rows)


In [4]:
# Step 4: Check all conversion stages using a single Munsell notation.
example_code = '5PB 7/4'
xyY_C, XYZ_C, XYZ_D65, Lab_D65, RGB_sRGB, RGB8 = munsell_to_d65_colourimetry(example_code)

print('Munsell code:', example_code)
print('xyY under Illuminant C:', np.round(xyY_C, 6))
print('XYZ under Illuminant C:', np.round(XYZ_C, 6))
print('XYZ adapted to D65:', np.round(XYZ_D65, 6))
print('CIE Lab (D65):', np.round(Lab_D65, 3))
print('Encoded sRGB (0-255):', RGB8)


Munsell code: 5PB 7/4
xyY under Illuminant C: [ 0.2773    0.2828    0.419854]
XYZ under Illuminant C: [ 0.411688  0.419854  0.65309 ]
XYZ adapted to D65: [ 0.396997  0.419084  0.601787]
CIE Lab (D65): [ 70.808  -0.415 -14.451]
Encoded sRGB (0-255): [158 175 199]


In [5]:
# Step 5: Export results as a DOCX table with sRGB colour rectangles.
def add_colour_table_docx(dataframe: pd.DataFrame, docx_path, title: str):
    """Create a DOCX report from D65 Lab and sRGB columns."""
    docx_path = Path(docx_path)
    document = Document()
    document.add_heading(title, level=1)
    document.add_paragraph(
        'Munsell coordinates under Illuminant C were adapted to D65 with CAT02. '
        'Lab is reported as CIE L*a*b*(D65); RGB is encoded sRGB.'
    )

    headers = ['Item', 'Munsell code', 'L*_D65', 'a*_D65', 'b*_D65', 'R_sRGB', 'G_sRGB', 'B_sRGB', 'Colour sample']
    table = document.add_table(rows=1, cols=len(headers))
    table.style = 'Table Grid'
    for column_index, header in enumerate(headers):
        table.rows[0].cells[column_index].text = header

    for row_index, row in dataframe.iterrows():
        cells = table.add_row().cells
        cells[0].text = str(row['Item'])
        cells[1].text = str(row['Munsell code'])
        cells[2].text = f"{row['L*_D65']:.3f}"
        cells[3].text = f"{row['a*_D65']:.3f}"
        cells[4].text = f"{row['b*_D65']:.3f}"
        cells[5].text = str(int(row['R_sRGB']))
        cells[6].text = str(int(row['G_sRGB']))
        cells[7].text = str(int(row['B_sRGB']))

        rgb = (int(row['R_sRGB']), int(row['G_sRGB']), int(row['B_sRGB']))
        swatch_path = OUTPUT_DIRECTORY / f'swatch_{row_index + 1}.png'
        Image.new('RGB', (240, 80), rgb).save(swatch_path)
        cells[8].paragraphs[0].add_run().add_picture(str(swatch_path), width=Inches(1.25), height=Inches(0.42))

    document.save(docx_path)
    return docx_path


In [6]:
# Step 6: Convert the reference project CSV and write CSV/DOCX result files.
REFERENCE_CSV_URL = 'https://raw.githubusercontent.com/postnicov/ResazurinResorufin/refs/heads/main/Data/MilkMunsellCodes.csv'

df_input = pd.read_csv(REFERENCE_CSV_URL)
if len(df_input.columns) != 2:
    raise ValueError('Reference CSV must contain exactly two columns.')

df_input.columns = ['Item', 'Munsell code']
df_input['Munsell code'] = df_input['Munsell code'].astype(str).str.strip()

df_converted = convert_munsell_dataframe(df_input)
csv_path = OUTPUT_DIRECTORY / 'MilkMunsell_RGB_Lab_D65_adapted.csv'
docx_path = OUTPUT_DIRECTORY / 'MilkMunsell_RGB_Lab_D65_adapted.docx'
df_converted.to_csv(csv_path, index=False)
add_colour_table_docx(df_converted, docx_path, 'Munsell to CIE Lab (D65) and sRGB')

print(f'CSV written to: {csv_path}')
print(f'DOCX written to: {docx_path}')
display(df_converted)


CSV written to: output/MilkMunsell_RGB_Lab_D65_adapted.csv
DOCX written to: output/MilkMunsell_RGB_Lab_D65_adapted.docx


,Item,Munsell code,x_C,y_C,Y_C,X_C,Y_C_XYZ,Z_C,X_D65,Y_D65,Z_D65,L*_D65,a*_D65,b*_D65,R_sRGB,G_sRGB,B_sRGB,R_sRGB_normalised,G_sRGB_normalised,B_sRGB_normalised
0,L6,5PB 7/4,0.27730,0.282800,0.419854,0.411688,0.419854,0.653090,0.396997,0.419084,0.601787,70.807747,-0.415224,-14.451164,158,175,199,0.619875,0.684652,0.781357
1,L4,10PB 7/5.5,0.28035,0.265925,0.419854,0.442629,0.419854,0.716361,0.427073,0.418487,0.660140,70.766461,8.972431,-19.664199,172,169,209,0.673762,0.664135,0.818800
2,L3,5P 7/4,0.30090,0.283100,0.419854,0.446252,0.419854,0.616952,0.431953,0.418942,0.568454,70.797890,10.290024,-11.380203,183,168,194,0.716848,0.658330,0.760767
3,L2,10P 7/8,0.32560,0.265400,0.419854,0.515088,0.419854,0.647024,0.500273,0.418156,0.596183,70.743593,29.807819,-14.050675,215,154,199,0.844015,0.604992,0.781367
4,LP6,10B 9/2,0.29490,0.307600,0.766956,0.735290,0.766956,0.991108,0.711050,0.766698,0.913081,90.169396,-3.727432,-5.538223,214,229,237,0.838083,0.899856,0.930115
5,LP5,2.5PB 9/2,0.29750,0.306300,0.766956,0.744921,0.766956,0.992060,0.720651,0.766604,0.913958,90.165060,-1.675116,-5.606020,218,228,237,0.854965,0.895067,0.930831
6,LP4,7.5PB 9/2,0.30150,0.305200,0.766956,0.757658,0.766956,0.988348,0.733410,0.766504,0.910533,90.160453,1.019774,-5.378022,224,227,237,0.877921,0.888576,0.929433
7,LP3,5P 9/2,0.30670,0.306000,0.766956,0.768710,0.766956,0.970726,0.744665,0.766488,0.894278,90.159722,3.356948,-4.251394,230,225,235,0.900816,0.882583,0.921371
8,LP2,2.5RP 9/2,0.31490,0.310800,0.766956,0.777073,0.766956,0.923654,0.753609,0.766642,0.850863,90.166823,5.164607,-1.157946,236,224,229,0.925795,0.877283,0.898751
9,LP1,5RP 9/3,0.32365,0.309300,0.766956,0.802539,0.766956,0.910156,0.779197,0.766472,0.838411,90.158990,10.377814,-0.268448,247,220,228,0.968032,0.863751,0.892801


In [7]:
# Step 7: Download outputs when the notebook is running in Google Colab.
if files is None:
    print('Google Colab is not active; files are in the output directory.')
else:
    files.download(str(csv_path))
    files.download(str(docx_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>